# Landsat vegetation indices from USGS ESPA, resized to 4 km by bilinear interpolation, Ogallala aquifer

USGS's **ESPA On Demand Interface** (https://espa.cr.usgs.gov) computes Landsat spectral indices per scene on
its own servers and can resize them before delivery. This notebook orders that work through ESPA's API,
waits, downloads the small results, and packs them into CF NetCDF files on Drive. No Earth Engine, and
only a few megabytes per scene to download.

| | |
|---|---|
| Output | `MyDrive/MSUGWB/LANDSAT_ESPA_indices_4km/LANDSAT_ESPA_indices_4km_p<path>r<row>_Ogallala.nc` |
| One record | one Landsat scene (Landsat 4, 5, 7, 8, 9; Collection 2 Level-2, Tier 1) |
| Grid | gridMET's 1/24° cells (about 4 km), same cells as `GRIDMET_Ogallala_lab.nc`, whole window kept |
| Variables | `ndvi`, `evi`, `savi`, `msavi`, `ndmi` as ESPA's 16-bit integers with their scale factor; `qa_pixel`, `qa_radsat`; `roi_mask`, `roi_fraction` |

ESPA also offers `sr_nbr`, `sr_nbr2` and `sr_ndsi`; add them to `INDICES` in Cell 2 if wanted.

**Before the first run**

1. Colab → key icon (Secrets) → add `ERS_USERNAME` and `ERS_PASSWORD` (your EarthExplorer account; register
   at https://ers.cr.usgs.gov/register) and allow notebook access. The login is only read from there.
2. *Run all* with `TEST_MODE = True`: one order of 6 scenes into `.../_test/`. This is the first real
   contact with ESPA, so it also shows whether ESPA's files land exactly on the requested grid.
3. Set `TEST_MODE = False` and *Run all*. ESPA needs hours to days per order. The notebook stops cleanly
   after `MAX_RUN_HOURS`; run all again later (daily is fine) and it collects whatever has finished.

**How it works** (order → wait → download → check → write → delete → repeat)

* The scene list comes from USGS's public Landsat catalogue (STAC) and is cached on Drive.
* Scenes are ordered in batches of 2,000, never more than 9,000 open at once (ESPA's limits are 5,000 per
  order and 10,000 open). Each order asks for: the chosen indices, geographic projection, 1/24° pixels,
  bilinear resampling, and the aquifer's gridMET window as image extents, so every scene arrives on the
  same grid.
* `espa_orders.json` on Drive records which scene is in which order. A new session reads it, asks ESPA
  for the state of the open orders and carries on; nothing is ordered twice.
* Each archive is checked against ESPA's own MD5, its GeoTIFFs must sit exactly on the grid, and every
  record is read back from the NetCDF and compared before the archive is deleted. Files are copied to
  Drive hourly and whenever a run stops; at least 20 GB of scratch stays free.
* Scenes finish in no particular order, so records are appended as they arrive and put in time order
  when a path/row is complete.
* Scenes ESPA reports as unavailable are logged with ESPA's reason. Finished orders that ESPA has
  already removed are ordered again.

**What you should know about these values**

* **Clouds are not removed.** ESPA's index files carry no quality screening, and resizing blends cloudy
  and clear 30 m pixels into one 4 km value. `qa_pixel` is stored, but ESPA resamples quality bands by
  nearest neighbour, so it describes one 30 m pixel per cell, not the cell. `MAX_CLOUD_COVER` can limit
  the order to clearer scenes.
* **What "bilinear" does at this reduction is USGS's implementation.** The user guide names the method
  but not whether a 4 km value draws on all of the roughly 17,000 pixels under it or on a few around the
  cell centre. Compare one scene with `Landsat_Indices_Ogallala.ipynb` (exact 500 m means) before relying
  on it.
* Values are stored exactly as delivered: 16-bit integers, scale factor 0.0001, fill −9999, and 20000
  where ESPA flags saturation. Nothing is masked, composited or gap-filled here.
* Not yet run against the real service. The ordering logic was tested against a stand-in built from
  ESPA's API documentation; the names of the files inside ESPA's archives and the exact grid it returns
  are confirmed only by the rehearsal in step 2.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q netCDF4 rasterio geopandas shapely pyproj requests 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE     = True       # True -> one small order into .../_test/ ; False -> the whole record
TEST_SCENES   = 6          # scenes in the rehearsal order, spread over the windows below
TEST_TEMPORAL = [("1990-06-01", "1990-09-01"), ("2005-06-01", "2005-09-01"), ("2022-06-01", "2022-09-01")]

ESPA_API = "https://espa.cr.usgs.gov/api/v1"                    # USGS ESPA on-demand ordering service
STAC_URL = "https://landsatlook.usgs.gov/stac-server/search"    # USGS's public Landsat catalogue (no login)
SATELLITES = ["LANDSAT_4", "LANDSAT_5", "LANDSAT_7", "LANDSAT_8", "LANDSAT_9"]
TIER       = "T1"          # Tier 1 only
DATE_START = None          # None -> first acquisition; or "YYYY-MM-DD"
DATE_END   = None          # None -> now
MAX_CLOUD_COVER = 100      # order only scenes whose USGS scene cloud cover (%) is at most this; 100 = all
ONLY_PATHROWS   = None     # None -> every path/row; or e.g. ["031034", "030034"]
SKIP_SCENES     = []       # scene ids never to order

# ---- what ESPA is asked to make ----
INDICES = ["sr_ndvi", "sr_evi", "sr_savi", "sr_msavi", "sr_ndmi"]   # ESPA also offers sr_nbr, sr_nbr2, sr_ndsi
ORDER_QA   = True          # also order the resized Level-2 input ("l1") and keep its QA_PIXEL and QA_RADSAT
RESAMPLING = "bil"         # ESPA: "nn" nearest neighbour, "bil" bilinear, "cc" cubic convolution
OUTPUT_FORMAT = "gtiff"

# ---- the 4 km grid: gridMET's 1/24 degree cells, so the files stack on GRIDMET_Ogallala_lab.nc ----
PIXEL_DEG  = 1.0 / 24.0
GRID_WEST  = -105.95416663333334   # western edge of the first column (gridMET's own grid)
GRID_NORTH = 43.67083333333334     # northern edge of the first row
GRID_NX, GRID_NY = 233, 287        # the aquifer's window of the gridMET grid

# ---- ordering and waiting ----
SCENES_PER_ORDER = 2000    # ESPA allows 5,000 per order; smaller orders are delivered sooner
MAX_OPEN_SCENES  = 9000    # ESPA allows 10,000 scenes in processing per user
POLL_MINUTES     = 10      # wait between looks at the open orders
MAX_RUN_HOURS    = 11.0    # stop cleanly before Colab's session limit; run again to continue

N_DOWNLOAD_THREADS = 10    # archives downloaded at a time
COMPRESS_LEVEL = 1         # zlib level of the CF .nc (lossless)
MIN_FREE_GB       = 20.0   # scratch space that must stay free at all times
DISK_MAX_WAIT_S   = 2 * 3600
BACKUP_INTERVAL_S = 3600   # copy changed files to Drive at least this often
MAX_RETRIES       = 4      # extra attempts per request or download
READBACK_CHECKS   = 3      # records per file re-read and compared with the delivery (all in TEST_MODE)

MANIFEST_MAX_AGE_DAYS = 7  # refresh the cached scene list when older than this
REBUILD_MANIFEST      = False
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, USGS login, ESPA client ═══════════════
import re, gc, io, sys, json, gzip, glob, time, math, shutil, tarfile, hashlib, logging, threading, warnings
import xml.etree.ElementTree as ET
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import shapely
import shapely.geometry
import rasterio
import rasterio.features
from rasterio.io import MemoryFile
from affine import Affine
import requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("espa")
for _n in ("urllib3.connectionpool", "rasterio", "pyogrio"):
    logging.getLogger(_n).setLevel(logging.ERROR)

# ---------- names and paths (Drive = durable, scratch = fast and disposable) ----------
ROI_LABEL   = "Ogallala"
PRODUCT_TAG = "LANDSAT_ESPA_indices_4km"
_SUFFIX     = "_test" if TEST_MODE else ""
DRIVE_ROOT  = "/content/drive/MyDrive/MSUGWB"
DRIVE_OUT   = f"{DRIVE_ROOT}/{PRODUCT_TAG}" + ("/_test" if TEST_MODE else "")
SCRATCH     = "/content/espa_scratch" + _SUFFIX
RAW_DIR     = f"{SCRATCH}/raw"                  # downloaded archives, deleted once their scene is written
NC_DIR      = f"{SCRATCH}/nc"
DISK_PATH   = "/content"

MANIFEST_NAME = "manifest_landsat_stac.json.gz"
LEDGER_NAME   = "espa_orders.json"              # which scene is in which ESPA order, and how it ended
LOG_FAILED    = "failed_scenes.jsonl"
LOG_SKIPPED   = "skipped_scenes.jsonl"
LOG_FILES     = "pathrow_summary.jsonl"
for _d in (DRIVE_OUT, RAW_DIR, NC_DIR):
    os.makedirs(_d, exist_ok=True)

GRID_SOUTH = GRID_NORTH - GRID_NY * PIXEL_DEG
GRID_EAST  = GRID_WEST + GRID_NX * PIXEL_DEG
TIME_UNITS, TIME_CALENDAR = "seconds since 2000-01-01 00:00:00", "proleptic_gregorian"
EPOCH = datetime(2000, 1, 1)
SAT_NUMBER = {"LANDSAT_4": 4, "LANDSAT_5": 5, "LANDSAT_7": 7, "LANDSAT_8": 8, "LANDSAT_9": 9}
INDEX_NAMES = {"sr_ndvi": "Normalized Difference Vegetation Index", "sr_evi": "Enhanced Vegetation Index",
               "sr_savi": "Soil Adjusted Vegetation Index", "sr_msavi": "Modified Soil Adjusted Vegetation Index",
               "sr_ndmi": "Normalized Difference Moisture Index", "sr_nbr": "Normalized Burn Ratio",
               "sr_nbr2": "Normalized Burn Ratio 2", "sr_ndsi": "Normalized Difference Snow Index"}
QA_LAYERS = ["qa_pixel", "qa_radsat"] if ORDER_QA else []
LAYERS    = [p[3:] for p in INDICES] + QA_LAYERS          # variable names: ndvi, evi, ..., qa_pixel, qa_radsat
# what the spectral-index files hold according to USGS; replaced by each delivery's own metadata when present
INDEX_DEFAULTS = dict(scale_factor=0.0001, fill_value=-9999, saturate_value=20000)


class PipelineStop(RuntimeError):
    """Clean stop: everything written so far is safe, re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


class NotThere(IOError):
    """ESPA no longer has the file (finished orders are removed after a while): order the scene again."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"


def parse_iso(s):
    s = s.rstrip("Z")[:26]
    return datetime.strptime(s, "%Y-%m-%dT%H:%M:%S.%f" if "." in s else "%Y-%m-%dT%H:%M:%S")


# ---------- USGS login: only ever read from Colab Secrets or the environment ----------
def ers_credentials():
    user = pw = None
    try:
        from google.colab import userdata
        for key in ("ERS_USERNAME", "ERS_PASSWORD"):
            try:
                val = userdata.get(key)
            except Exception:
                val = None
            if key == "ERS_USERNAME":
                user = val
            else:
                pw = val
    except Exception:
        pass
    user, pw = user or os.environ.get("ERS_USERNAME"), pw or os.environ.get("ERS_PASSWORD")
    if not (user and pw):
        raise PipelineStop("No USGS login found. In Colab: key icon (Secrets) -> add ERS_USERNAME and ERS_PASSWORD "
                           "(the EarthExplorer account) and allow notebook access. Nothing is typed into the notebook.")
    return user, pw


class Espa:
    """Thin client of ESPA's ordering API (https://espa.cr.usgs.gov/static/docs/api-resources-list.html)."""

    def __init__(self, base, auth):
        self.base, self.s = base.rstrip("/"), requests.Session()
        self.s.auth = auth                       # HTTP basic authentication, sent to the ESPA host only

    def call(self, method, path, body=None):
        last = None
        for attempt in range(1 + MAX_RETRIES):
            try:
                r = self.s.request(method, f"{self.base}/{path}", json=body, timeout=180)
                if r.status_code in (401, 403):
                    raise PipelineStop(f"ESPA refused the login (HTTP {r.status_code}). Check the Colab Secrets "
                                       "ERS_USERNAME / ERS_PASSWORD, and that the account can sign in at "
                                       "https://espa.cr.usgs.gov/ in a browser.")
                if r.status_code >= 500 or r.status_code == 429:
                    raise IOError(f"HTTP {r.status_code}")
                try:
                    data = r.json()
                except ValueError:
                    raise IOError(f"HTTP {r.status_code}, reply is not JSON: {r.text[:200]}")
                msgs = data.get("messages", {}) if isinstance(data, dict) else {}
                if msgs.get("warnings"):
                    log.warning("ESPA warning on %s: %s", path.split("/")[0], str(msgs["warnings"])[:300])
                if msgs.get("errors") or r.status_code >= 400:
                    raise ValueError(f"ESPA rejected {method} {path.split('/')[0]} (HTTP {r.status_code}): "
                                     f"{str(msgs.get('errors') or data)[:600]}")
                return data
            except (PipelineStop, ValueError):
                raise
            except Exception as e:
                last = e
                if attempt < MAX_RETRIES:
                    time.sleep(min(15 * 2 ** attempt, 300))
        raise PipelineStop(f"ESPA did not answer {method} {path.split('/')[0]} after {1 + MAX_RETRIES} attempts: {last}")

    def user(self):
        return self.call("GET", "user")

    def available(self, ids):
        return self.call("GET", "available-products", {"inputs": list(ids)})

    def order(self, body):
        return self.call("POST", "order", body)["orderid"]

    def items(self, orderid):
        data = self.call("GET", f"item-status/{orderid}")
        return data.get(orderid) or next(iter(data.values()), [])

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))

In [ ]:
# ═══════════════ CELL 5: the 4 km grid and aquifer mask, scene list from USGS's catalogue, path/row windows ═══════════════
GRID_TRANSFORM = Affine(PIXEL_DEG, 0.0, GRID_WEST, 0.0, -PIXEL_DEG, GRID_NORTH)
SUB = 8      # roi_fraction is measured on an 8 x 8 sub-grid of every cell


def load_roi(shp_path):
    """Aquifer polygon on the grid: roi_mask (cell centre inside), roi_fraction (share of the cell
    inside), and a rough outline used only as the spatial filter of the catalogue search."""
    gdf = gpd.read_file(shp_path)
    if gdf.crs is None:
        raise ValueError(f"{shp_path} has no CRS")
    g = gdf.to_crs(4326).geometry
    poly = g.union_all() if hasattr(g, "union_all") else g.unary_union
    if not poly.is_valid:
        poly = poly.buffer(0)
    mask = rasterio.features.geometry_mask([poly], out_shape=(GRID_NY, GRID_NX), transform=GRID_TRANSFORM,
                                           invert=True, all_touched=False)
    fine = rasterio.features.geometry_mask([poly], out_shape=(GRID_NY * SUB, GRID_NX * SUB), invert=True,
                                           transform=GRID_TRANSFORM * Affine.scale(1.0 / SUB), all_touched=False)
    frac = fine.reshape(GRID_NY, SUB, GRID_NX, SUB).mean(axis=(1, 3)).astype("f4")
    rough = poly.buffer(0.15).simplify(0.10, preserve_topology=True)
    if rough.geom_type != "Polygon":
        rough = rough.convex_hull
    return mask, frac, shapely.geometry.mapping(shapely.geometry.Polygon(rough.exterior))


ROI_MASK, ROI_FRACTION, ROI_GEOJSON = load_roi(SHP_PATH)
if not ROI_MASK.any():
    raise PipelineStop("The aquifer polygon does not touch the grid - check SHP_PATH and the grid settings.")
log.info("Grid: %d x %d cells of 1/24 degree, west %.6f north %.6f | %d cells have their centre in the aquifer",
         GRID_NY, GRID_NX, GRID_WEST, GRID_NORTH, int(ROI_MASK.sum()))

_ID_RE = re.compile(r"^(L[A-Z]\d{2}_L2S[PR]_\d{6}_\d{8}_\d{8}_02_T\d)(?:_SR)?$")


def _stac_pages(body):
    """All items of one catalogue search, following the 'next' links."""
    items = []
    while True:
        last = None
        for attempt in range(1 + MAX_RETRIES):
            try:
                r = requests.post(STAC_URL, json=body, timeout=120)
                if r.status_code != 200:
                    raise IOError(f"HTTP {r.status_code}: {r.text[:200]}")
                page = r.json()
                break
            except Exception as e:
                last = e
                time.sleep(min(10 * 2 ** attempt, 120))
        else:
            raise PipelineStop(f"USGS catalogue search failed: {last}")
        feats = page.get("features", [])
        items += feats
        nxt = [l for l in page.get("links", []) if l.get("rel") == "next"]
        if not feats or not nxt:
            return items
        body = nxt[0].get("body") or body


def _scene_record(f):
    m = _ID_RE.match(str(f.get("id", "")))
    p = f.get("properties", {})
    if not m or "landsat:wrs_path" not in p or not f.get("bbox"):
        return None
    return dict(id=m.group(1), t=p["datetime"], platform=str(p.get("platform", "")),
                path=int(p["landsat:wrs_path"]), row=int(p["landsat:wrs_row"]),
                cloud=float(p.get("eo:cloud_cover", -1) if p.get("eo:cloud_cover") is not None else -1),
                sun_elev=float(p.get("view:sun_elevation") or float("nan")),
                tier=str(p.get("landsat:collection_category", "")), bbox=[float(v) for v in f["bbox"]])


def get_manifest():
    """Every Collection 2 Level-2 scene of the chosen satellites and tier over the aquifer's outline."""
    if TEST_MODE:
        windows = list(TEST_TEMPORAL)
    else:
        y0 = int((DATE_START or "1982-01-01")[:4])
        y1 = int((DATE_END or f"{utcnow().year + 1}-01-01")[:4])
        windows = [(max(f"{y}-01-01", DATE_START or ""), min(f"{y + 1}-01-01", DATE_END or "9999")) for y in range(y0, y1 + 1)]
        windows = [w for w in windows if w[0] < w[1]]
    sig = json.dumps(dict(sats=SATELLITES, tier=TIER, windows=windows), sort_keys=True)
    path = os.path.join(DRIVE_OUT, MANIFEST_NAME)
    if os.path.exists(path) and not REBUILD_MANIFEST:
        try:
            with gzip.open(path, "rt") as f:
                m = json.load(f)
            age = (utcnow() - parse_iso(m["created"])).total_seconds() / 86400
            if m.get("signature") == sig and age <= MANIFEST_MAX_AGE_DAYS:
                log.info("Scene list from Drive: %d scenes, %.1f days old", len(m["scenes"]), age)
                return m["scenes"]
        except Exception as e:
            log.warning("Cached scene list unreadable (%s) - searching again", e)
    t0 = time.time()

    def one(w):
        return _stac_pages(dict(collections=["landsat-c2l2-sr"], intersects=ROI_GEOJSON, limit=500,
                                datetime=f"{w[0]}T00:00:00Z/{w[1]}T00:00:00Z",
                                query={"landsat:collection_category": {"eq": TIER},
                                       "platform": {"in": SATELLITES}}))

    recs, odd = {}, 0
    with ThreadPoolExecutor(min(6, len(windows))) as ex:
        for feats in ex.map(one, windows):
            for f in feats:
                r = _scene_record(f)
                if r is None or r["platform"] not in SAT_NUMBER:
                    odd += 1
                else:
                    recs[r["id"]] = r
    scenes = sorted(recs.values(), key=lambda r: (r["t"], r["id"]))
    local = os.path.join(NC_DIR, MANIFEST_NAME)
    with gzip.open(local, "wt") as f:
        json.dump(dict(created=iso(utcnow()), signature=sig, scenes=scenes), f)
    robust_drive_copy(local, path)
    os.remove(local)
    log.info("Scene list from USGS's catalogue: %d scenes (%s), %d entries not usable, %.0f s", len(scenes),
             ", ".join(f"L{n} {sum(1 for r in scenes if r['platform'] == s)}" for s, n in SAT_NUMBER.items()),
             odd, time.time() - t0)
    return scenes


def _cells(bbox, pad=1):
    """Rows/cols of the grid touched by a lon/lat box (west, south, east, north), clipped to the grid."""
    w, s, e, n = bbox
    c0 = max(0, int(math.floor((w - GRID_WEST) / PIXEL_DEG)) - pad)
    c1 = min(GRID_NX, int(math.ceil((e - GRID_WEST) / PIXEL_DEG)) + pad)
    r0 = max(0, int(math.floor((GRID_NORTH - n) / PIXEL_DEG)) - pad)
    r1 = min(GRID_NY, int(math.ceil((GRID_NORTH - s) / PIXEL_DEG)) + pad)
    return r0, r1, c0, c1


def build_pathrows(scenes):
    """Group the scenes by WRS-2 path/row. Each path/row gets one file and one fixed window of the
    grid: the union of its scenes' footprints. Path/rows that hold no aquifer cell are left out."""
    groups = {}
    for r in scenes:
        if r["id"] in SKIP_SCENES or (0 <= MAX_CLOUD_COVER < 100 and r["cloud"] > MAX_CLOUD_COVER):
            continue
        groups.setdefault((r["path"], r["row"]), []).append(r)
    out = []
    for (p, w_), rs in groups.items():
        b = np.array([r["bbox"] for r in rs])
        r0, r1, c0, c1 = _cells((b[:, 0].min(), b[:, 1].min(), b[:, 2].max(), b[:, 3].max()))
        if r0 >= r1 or c0 >= c1 or not ROI_MASK[r0:r1, c0:c1].any():
            continue
        out.append(dict(key=f"{p:03d}{w_:03d}", path=p, row=w_, win=(r0, r1, c0, c1),
                        n_roi=int(ROI_MASK[r0:r1, c0:c1].sum()), scenes=sorted(rs, key=lambda r: (r["t"], r["id"]))))
    out.sort(key=lambda P: (-P["n_roi"], P["key"]))
    return out

In [ ]:
# ═══════════════ CELL 6: ESPA orders (ledger on Drive, order text, submit, poll) ═══════════════
WANTED = list(INDICES) + (["l1"] if ORDER_QA else [])
OPEN_STATES = ("ordered",)                 # ledger states: ordered -> written | unavailable | reorder
ORDER_OPTIONS = dict(
    format=OUTPUT_FORMAT, resampling_method=RESAMPLING,
    projection={"lonlat": None},                                           # geographic, WGS 84
    resize=dict(pixel_size=PIXEL_DEG, pixel_size_units="dd"),
    image_extents=dict(north=GRID_NORTH, south=GRID_SOUTH, east=GRID_EAST, west=GRID_WEST, units="dd"))
ERROR_GIVE_UP_HOURS = 48     # a scene ESPA still reports as 'error' this long after ordering is recorded as unavailable
_LEDGER_LOCK = threading.Lock()


def load_ledger():
    path = os.path.join(DRIVE_OUT, LEDGER_NAME)
    if os.path.exists(path):
        with open(path) as f:
            led = json.load(f)
        if led.get("options") != json.loads(json.dumps(dict(ORDER_OPTIONS, products=WANTED))):
            raise PipelineStop(f"{LEDGER_NAME} on Drive belongs to orders made with other settings (grid, resampling "
                               "or products). Restore the settings, or move the old output folder away.")
        return led
    return dict(options=dict(ORDER_OPTIONS, products=WANTED), orders={}, scenes={})


def save_ledger(led):
    """One write + fsync under a temporary name, then a rename: a dead runtime cannot leave half a ledger."""
    path = os.path.join(DRIVE_OUT, LEDGER_NAME)
    with _LEDGER_LOCK:
        with open(path + ".tmp", "w") as f:
            json.dump(led, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(path + ".tmp", path)


def submit_order(espa, led, ids, note):
    """Ask ESPA what it can make for these scenes, then order exactly that. Scenes ESPA cannot
    process, or for which it offers none of the requested indices, are recorded as unavailable."""
    avail = espa.available(ids)
    body, ordered, lost = {}, [], []
    for sensor, v in avail.items():
        if sensor == "not_implemented":
            lost += [(s, "ESPA: not implemented for this scene id") for s in v]
            continue
        offered = set(v.get("products", []))
        products = [p for p in WANTED if p in offered]
        if not any(p in offered for p in INDICES):
            lost += [(s, f"ESPA offers none of the requested indices for {sensor}") for s in v.get("inputs", [])]
            continue
        missing = [p for p in WANTED if p not in offered]
        if missing:
            log.warning("ESPA does not offer %s for %s (%d scenes) - ordered without", missing, sensor, len(v["inputs"]))
        body[sensor] = dict(inputs=list(v["inputs"]), products=products)
        ordered += list(v["inputs"])
    lost += [(s, "ESPA did not return this scene id") for s in set(ids) - set(ordered) - {s for s, _ in lost}]
    for s, why in lost:
        led["scenes"][s] = dict(state="unavailable", note=why)
    log_events(LOG_FAILED, [dict(scene=s, reason=why, stage="order") for s, why in lost])
    if not ordered:
        save_ledger(led)
        return None
    body.update(ORDER_OPTIONS, note=note)
    orderid = espa.order(body)
    led["orders"][orderid] = dict(submitted=iso(utcnow()), n=len(ordered), closed=False)
    for s in ordered:
        led["scenes"][s] = dict(state="ordered", order=orderid)
    save_ledger(led)
    log.info("Ordered %d scenes from ESPA: %s%s", len(ordered), orderid,
             f" ({len(lost)} not available)" if lost else "")
    return orderid


def poll_orders(espa, led):
    """Look at every open order. Returns the finished items not yet written: [(scene, item)].
    Scenes ESPA gave up on are recorded; an order with nothing left open is closed."""
    ready, lost, counts = [], [], {}
    for orderid, o in led["orders"].items():
        if o.get("closed"):
            continue
        still_open = 0
        age_h = (utcnow() - parse_iso(o["submitted"])).total_seconds() / 3600
        try:
            items = espa.items(orderid)
        except ValueError as e:                       # ESPA no longer knows the order: its scenes are ordered again
            log.warning("Order %s can no longer be read (%s) - its open scenes will be ordered again", orderid, str(e)[:150])
            for scene, rec in led["scenes"].items():
                if rec.get("order") == orderid and rec["state"] == "ordered":
                    rec["state"] = "reorder"
            o["closed"] = True
            continue
        for it in items:
            scene, status = str(it.get("name", "")), str(it.get("status", "")).lower()
            rec = led["scenes"].get(scene)
            if rec is None or rec.get("order") != orderid or rec["state"] != "ordered":
                continue
            counts[status] = counts.get(status, 0) + 1
            if status == "complete" and it.get("product_dload_url"):
                ready.append((scene, dict(it, order=orderid)))
                still_open += 1
            elif status in ("unavailable", "cancelled", "purged") or (status == "error" and age_h > ERROR_GIVE_UP_HOURS):
                why = f"ESPA status '{status}': {str(it.get('note', ''))[:300]}"
                led["scenes"][scene] = dict(state="reorder" if status == "purged" else "unavailable", note=why, order=orderid)
                if status != "purged":
                    lost.append(dict(scene=scene, reason=why, stage="processing", order=orderid))
            else:
                still_open += 1                      # submitted, oncache, onorder, queued, processing, error, retry
        if still_open == 0:
            o["closed"] = True
    log_events(LOG_FAILED, lost)
    if lost or any(o.get("closed") for o in led["orders"].values()):
        save_ledger(led)
    return ready, counts

In [ ]:
# ═══════════════ CELL 7: download a finished scene, check it, read it onto the grid ═══════════════
def md5_of(path, block=4 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def download_item(scene, item):
    """ESPA archive (.tar.gz) -> scratch, checked against ESPA's own MD5 file. The download links
    are plain HTTPS links; the USGS password is not sent to them."""
    url, sum_url = item["product_dload_url"], item.get("cksum_download_url") or ""
    path = os.path.join(RAW_DIR, f"{scene}.tar.gz")
    last = None
    for attempt in range(1 + MAX_RETRIES):
        try:
            with requests.get(url, stream=True, timeout=300) as r:
                if r.status_code in (404, 410):
                    raise NotThere(f"{scene}: ESPA no longer has {os.path.basename(url)} (HTTP {r.status_code})")
                if r.status_code != 200:
                    raise IOError(f"HTTP {r.status_code}")
                with open(path + ".part", "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        f.write(chunk)
            want = None
            if sum_url:
                s = requests.get(sum_url, timeout=120)
                if s.status_code == 200 and re.match(r"^[0-9a-fA-F]{32}\b", s.text.strip()):
                    want = s.text.strip().split()[0].lower()
            got = md5_of(path + ".part")
            if want and got != want:
                raise IOError(f"MD5 {got} differs from ESPA's {want}")
            with tarfile.open(path + ".part") as tar:            # must at least open as an archive
                if not tar.getnames():
                    raise IOError("empty archive")
            os.replace(path + ".part", path)
            return dict(path=path, bytes=os.path.getsize(path), md5=got, md5_checked=bool(want))
        except NotThere:
            raise
        except Exception as e:
            last = e
            if attempt < MAX_RETRIES:
                time.sleep(min(10 * 2 ** attempt, 120))
    raise IOError(f"{scene}: download failed after {1 + MAX_RETRIES} attempts: {last}")


def _layer_of(name):
    """Which requested layer a file inside the archive holds, judged by the end of its name."""
    base = os.path.basename(name).lower()
    if not base.endswith((".tif", ".tiff")):
        return None
    stem = base.rsplit(".", 1)[0]
    for layer in LAYERS:
        ends = (f"_{layer}",) if not layer.startswith("qa_") else (f"_{layer}", f"_{layer[3:]}_qa")
        if stem.endswith(ends):
            return layer
    return None


def _xml_bands(tar):
    """Per-band facts from ESPA's own metadata file, when the archive has one: fill, scale, saturation."""
    out = {}
    for m in tar.getmembers():
        if not m.name.lower().endswith(".xml") or "mtl" in m.name.lower():
            continue
        try:
            root = ET.fromstring(tar.extractfile(m).read())
        except Exception:
            continue
        for el in root.iter():
            if not el.tag.lower().endswith("band"):
                continue
            a = {k.lower(): v for k, v in el.attrib.items()}
            name = a.get("name", "").lower()
            key = name[3:] if name.startswith("sr_") else name
            key = {"pixel_qa": "qa_pixel", "radsat_qa": "qa_radsat"}.get(key, key)
            if key in LAYERS:
                out[key] = {k: a[k] for k in ("fill_value", "scale_factor", "saturate_value", "data_type") if k in a}
    return out


def _grid_offset(ds, name):
    """Row/col of the file's upper-left corner on the 4 km grid. The file must be geographic, have
    the grid's cell size, and sit a whole number of cells from the grid's corner."""
    t = ds.transform
    if ds.crs is None or not ds.crs.is_geographic or t.b or t.d:
        raise DataIntegrityError(f"{name}: not a north-up geographic raster (CRS {ds.crs})")
    drift = max(abs(t.a - PIXEL_DEG) * GRID_NX, abs(-t.e - PIXEL_DEG) * GRID_NY) / PIXEL_DEG
    col, row = (t.c - GRID_WEST) / PIXEL_DEG, (GRID_NORTH - t.f) / PIXEL_DEG
    if drift > 1e-3 or abs(col - round(col)) > 1e-3 or abs(row - round(row)) > 1e-3:
        raise DataIntegrityError(
            f"{name}: ESPA's grid does not line up with the requested one. Cell size {t.a!r} x {-t.e!r} (asked "
            f"{PIXEL_DEG!r}), corner {t.c!r}, {t.f!r} = {col:.4f} columns and {row:.4f} rows from the grid corner. "
            "Nothing is resampled here to hide that: adjust the grid settings to what ESPA delivers, or ask USGS.")
    return int(round(row)), int(round(col))


def read_delivery(tar_path, P):
    """One ESPA archive -> the requested layers on the path/row window, exactly as delivered."""
    r0, r1, c0, c1 = P["win"]
    arrays, meta, outside = {}, {}, 0
    with tarfile.open(tar_path) as tar:
        xml = _xml_bands(tar)
        for m in tar.getmembers():
            layer = _layer_of(m.name)
            if layer is None or layer in arrays:
                continue
            with MemoryFile(tar.extractfile(m).read()) as mem, mem.open() as ds:
                if ds.count != 1:
                    raise DataIntegrityError(f"{m.name}: {ds.count} bands in one file")
                row, col = _grid_offset(ds, m.name)
                data = ds.read(1)
                x = xml.get(layer, {})
                is_index = not layer.startswith("qa_")
                if ds.nodata is not None:
                    fill, fill_src = ds.nodata, "GeoTIFF nodata tag"
                elif "fill_value" in x:
                    fill, fill_src = float(x["fill_value"]), "ESPA metadata"
                else:
                    fill = INDEX_DEFAULTS["fill_value"] if is_index else (1 if layer == "qa_pixel" else 65535)
                    fill_src = "assumed (not stated in the delivery)"
                fill = data.dtype.type(fill)
                full = np.full((GRID_NY, GRID_NX), fill, dtype=data.dtype)
                a, b = max(row, 0), max(col, 0)
                e, f = min(row + data.shape[0], GRID_NY), min(col + data.shape[1], GRID_NX)
                if a < e and b < f:
                    full[a:e, b:f] = data[a - row:e - row, b - col:f - col]
                win = np.ascontiguousarray(full[r0:r1, c0:c1])
                if is_index:
                    outside = max(outside, int((full != fill).sum() - (win != fill).sum()))
                arrays[layer] = win
                meta[layer] = dict(dtype=str(data.dtype), fill=fill.item(), fill_source=fill_src, file=os.path.basename(m.name))
                if is_index:
                    for k in ("scale_factor", "saturate_value"):
                        meta[layer][k] = float(x[k]) if k in x else INDEX_DEFAULTS[k]
                        meta[layer][k + "_source"] = "ESPA metadata" if k in x else "USGS documentation (not stated in the delivery)"
    missing = [l for l in LAYERS if l not in arrays]
    if missing:
        with tarfile.open(tar_path) as tar:
            names = [os.path.basename(n) for n in tar.getnames()]
        raise DataIntegrityError(f"{os.path.basename(tar_path)}: no file for {missing}. The archive holds {names[:40]}")
    first = LAYERS[0]
    valid = arrays[first] != meta[first]["fill"]
    mask = ROI_MASK[r0:r1, c0:c1]
    stats = dict(valid_cells=int(valid.sum()), valid_roi_cells=int((valid & mask).sum()),
                 valid_cell_fraction=float((valid & mask).sum() / max(1, mask.sum())), outside_window=outside,
                 saturated=int((arrays[first] == meta[first].get("saturate_value", INDEX_DEFAULTS["saturate_value"])).sum()))
    return dict(arrays=arrays, meta=meta, stats=stats, archive=os.path.basename(tar_path))

In [ ]:
# ═══════════════ CELL 8: CF NetCDF writer, resume state, time ordering, read-back check, file audit ═══════════════
SETTINGS_SIGNATURE = json.dumps(dict(layers=LAYERS, resampling=RESAMPLING, pixel=PIXEL_DEG, west=GRID_WEST,
                                     north=GRID_NORTH, nx=GRID_NX, ny=GRID_NY, tier=TIER), sort_keys=True)
RESAMPLING_NAME = {"nn": "nearest neighbour", "bil": "bilinear interpolation", "cc": "cubic convolution"}
PER_SCENE = [("satellite", "i1"), ("cloud_cover", "f4"), ("sun_elevation", "f4"), ("valid_cell_fraction", "f4")]
PER_SCENE_TEXT = ["source_scene", "espa_order", "espa_archive"]


def pr_filename(key):
    return f"{PRODUCT_TAG}_p{key[:3]}r{key[3:]}_{ROI_LABEL}.nc"


def scene_seconds(r):
    return (parse_iso(r["t"]) - EPOCH).total_seconds()


def create_pr_nc(path, P, meta):
    """Create the empty CF file of one path/row on its window of the 4 km grid. Data types, fill
    values and scale factors are those of the first ESPA delivery for this path/row."""
    r0, r1, c0, c1 = P["win"]
    ny, nx = r1 - r0, c1 - c0
    lat = GRID_NORTH - (r0 + np.arange(ny) + 0.5) * PIXEL_DEG
    lon = GRID_WEST + (c0 + np.arange(nx) + 0.5) * PIXEL_DEG
    zkw = dict(zlib=COMPRESS_LEVEL > 0, complevel=max(COMPRESS_LEVEL, 1), shuffle=True)
    tmp = path + ".new"
    ds = nc4.Dataset(tmp, "w", format="NETCDF4")
    try:
        ds.createDimension("time", None)
        ds.createDimension("lat", ny)
        ds.createDimension("lon", nx)
        tv = ds.createVariable("time", "f8", ("time",), chunksizes=(512,))
        tv.units, tv.calendar, tv.standard_name, tv.axis = TIME_UNITS, TIME_CALENDAR, "time", "T"
        tv.long_name = "scene centre time (UTC)"
        la = ds.createVariable("lat", "f8", ("lat",))
        la.standard_name, la.units, la.axis, la.long_name = "latitude", "degrees_north", "Y", "latitude of cell centre"
        la[:] = lat
        lo = ds.createVariable("lon", "f8", ("lon",))
        lo.standard_name, lo.units, lo.axis, lo.long_name = "longitude", "degrees_east", "X", "longitude of cell centre"
        lo[:] = lon
        gm = ds.createVariable("crs", "i4")
        crs = pyproj.CRS.from_epsg(4326)
        for k, v in crs.to_cf().items():
            gm.setncattr(k, v if isinstance(v, (str, int, float)) else str(v))
        gm.spatial_ref = crs.to_wkt()
        gm.GeoTransform = f"{GRID_WEST + c0 * PIXEL_DEG} {PIXEL_DEG} 0 {GRID_NORTH - r0 * PIXEL_DEG} 0 {-PIXEL_DEG}"
        gm.grid_corner = np.array([GRID_WEST, GRID_NORTH], dtype="f8")      # west and north edge of the whole grid
        gm.grid_shape = np.array([GRID_NY, GRID_NX], dtype="i4")
        gm.grid_window = np.array(P["win"], dtype="i4")                     # row0, row1, col0, col1 stored here
        gm.long_name = "gridMET 1/24 degree grid (WGS 84), window of this path/row"
        mv = ds.createVariable("roi_mask", "i1", ("lat", "lon"), **zkw)
        mv.long_name = f"1 where the cell centre lies inside the {ROI_LABEL} aquifer polygon"
        mv.flag_values, mv.flag_meanings, mv.grid_mapping = np.array([0, 1], dtype="i1"), "outside_roi inside_roi", "crs"
        mv[:] = ROI_MASK[r0:r1, c0:c1].astype("i1")
        fv = ds.createVariable("roi_fraction", "f4", ("lat", "lon"), **zkw)
        fv.long_name, fv.units, fv.grid_mapping = f"share of the cell inside the {ROI_LABEL} aquifer polygon", "1", "crs"
        fv[:] = ROI_FRACTION[r0:r1, c0:c1]
        chunks = (1, ny, nx)
        how = RESAMPLING_NAME.get(RESAMPLING, RESAMPLING)
        for layer in LAYERS:
            m = meta[layer]
            dt = np.dtype(m["dtype"])
            v = ds.createVariable(layer, dt, ("time", "lat", "lon"), fill_value=dt.type(m["fill"]), chunksizes=chunks, **zkw)
            v.grid_mapping = "crs"
            v.fill_value_source = m["fill_source"]
            if layer.startswith("qa_"):
                v.long_name = ("Landsat pixel quality flags (QA_PIXEL)" if layer == "qa_pixel"
                               else "Landsat radiometric saturation flags (QA_RADSAT)")
                v.units = "1"
                v.comment = ("Bit flags of ONE 30 m pixel per cell: ESPA always resamples quality bands by nearest "
                             "neighbour. They describe that pixel, not the 4 km cell. QA_PIXEL bits: 0 fill, 1 dilated "
                             "cloud, 2 cirrus, 3 cloud, 4 cloud shadow, 5 snow, 6 clear, 7 water.")
            else:
                v.long_name = INDEX_NAMES.get("sr_" + layer, layer.upper()) + f" ({layer.upper()}), resized by ESPA"
                v.units = "1"
                v.scale_factor = np.float64(m["scale_factor"])          # CF packing: value = stored * scale_factor
                v.add_offset = np.float64(0.0)
                v.scale_factor_source = m["scale_factor_source"]
                v.saturate_value = dt.type(m["saturate_value"])
                v.saturate_value_source = m["saturate_value_source"]
                v.cell_methods = f"area: point (resampled from 30 m by USGS ESPA, {how})"
                v.comment = (f"USGS ESPA computed the index at 30 m from Collection 2 Level-2 surface reflectance and "
                             f"resized it to this grid by {how}. Stored integers are exactly ESPA's. Clouds, shadow "
                             "and snow are NOT masked: ESPA's indices carry no quality screening, so a cell can hold "
                             f"cloud-contaminated values. Stored value {int(m['saturate_value'])} marks saturation.")
                if QA_LAYERS:
                    v.ancillary_variables = " ".join(QA_LAYERS)
        for name, kind in PER_SCENE:
            v = ds.createVariable(name, kind, ("time",), chunksizes=(512,))
            v.long_name = dict(satellite="Landsat satellite number", cloud_cover="scene cloud cover reported by USGS",
                               sun_elevation="sun elevation at scene centre",
                               valid_cell_fraction="share of this file's aquifer cells that hold a value")[name]
            v.units = dict(satellite="1", cloud_cover="percent", sun_elevation="degree", valid_cell_fraction="1")[name]
        for name in PER_SCENE_TEXT:
            ds.createVariable(name, str, ("time",)).long_name = dict(
                source_scene="Landsat Collection 2 Level-2 product id", espa_order="ESPA order that produced the record",
                espa_archive="name of the ESPA archive the record was read from")[name]
        ds.Conventions = "CF-1.9"
        ds.title = (f"Landsat spectral indices resized to 4 km by USGS ESPA, {ROI_LABEL} aquifer, WRS-2 path "
                    f"{P['path']} row {P['row']}, every scene")
        ds.summary = ("One record per Landsat scene. USGS's ESPA on-demand service computed the indices at 30 m and "
                      f"resized them to the gridMET 1/24 degree grid by {how}; this file stores what ESPA delivered, "
                      "cell for cell, without masking, compositing or gap filling.")
        ds.institution = "U.S. Geological Survey EROS (ESPA on-demand products); packaged by the MSU groundwater project"
        ds.source = "USGS ESPA On Demand Interface, https://espa.cr.usgs.gov/ ; scene list from the USGS Landsat STAC catalogue"
        ds.references = ("Landsat Spectral Indices products courtesy of the U.S. Geological Survey Earth Resources "
                         "Observation and Science Center; ESPA On Demand Interface User Guide LSDS-1417")
        ds.history = f"{iso(utcnow())} created by Landsat_ESPA_Indices_4km_Ogallala.ipynb"
        ds.wrs_path, ds.wrs_row, ds.roi, ds.layers = np.int32(P["path"]), np.int32(P["row"]), ROI_LABEL, " ".join(LAYERS)
        ds.espa_order_options = json.dumps(dict(ORDER_OPTIONS, products=WANTED), sort_keys=True)
        ds.settings_signature = SETTINGS_SIGNATURE
        ds.processing_status = "in_progress"
        ds.committed_records = np.int32(0)
        ds.time_sorted = np.int8(1)
    finally:
        ds.close()
    os.replace(tmp, path)


def file_state(path):
    """Scenes held by a file, or None when it cannot be trusted (unreadable or a torn write)."""
    try:
        with nc4.Dataset(path, "r") as ds:
            if str(getattr(ds, "settings_signature", "")) != SETTINGS_SIGNATURE:
                raise PipelineStop(f"{os.path.basename(path)} was made with other settings (layers, grid or "
                                   "resampling). Restore them, or move the old files away and start again.")
            n = len(ds.variables["time"])
            if int(getattr(ds, "committed_records", -1)) != n:
                log.warning("%s: %d records but %s committed - not trusted", os.path.basename(path), n,
                            getattr(ds, "committed_records", "?"))
                return None
            return dict(n=n, scenes=set(str(s) for s in ds.variables["source_scene"][:]) if n else set(),
                        status=str(getattr(ds, "processing_status", "in_progress")), size=os.path.getsize(path))
    except PipelineStop:
        raise
    except Exception as e:
        log.warning("%s cannot be read (%s) - not trusted", os.path.basename(path), e)
        return None


class PRFile:
    """Single writer of one path/row file. Scenes arrive in the order ESPA finishes them, so
    records are appended as they come and put in time order when the path/row is finished."""

    def __init__(self, path):
        self.path, self.ds, self.dirty = path, None, False
        self.open()

    def open(self):
        self.ds = nc4.Dataset(self.path, "a")
        self.ds.set_auto_maskandscale(False)
        self.n = len(self.ds.variables["time"])
        self.pending = 0

    def check_meta(self, meta):
        for layer in LAYERS:
            v, m = self.ds.variables[layer], meta[layer]
            same = str(v.dtype) == m["dtype"] and v.getncattr("_FillValue") == np.dtype(m["dtype"]).type(m["fill"])
            if same and not layer.startswith("qa_"):
                same = float(v.scale_factor) == float(m["scale_factor"])
            if not same:
                raise DataIntegrityError(f"{os.path.basename(self.path)}: {layer} arrived as {m['dtype']} / fill "
                                         f"{m['fill']} / scale {m.get('scale_factor')}, the file holds {v.dtype} / "
                                         f"{v.getncattr('_FillValue')} / {getattr(v, 'scale_factor', None)}")

    def append(self, r, res, orderid):
        self.check_meta(res["meta"])
        i, v = self.n, self.ds.variables
        for layer in LAYERS:
            v[layer][i, :, :] = res["arrays"][layer]
        v["satellite"][i] = SAT_NUMBER[r["platform"]]
        v["cloud_cover"][i] = r["cloud"]
        v["sun_elevation"][i] = r["sun_elev"]
        v["valid_cell_fraction"][i] = res["stats"]["valid_cell_fraction"]
        v["source_scene"][i] = r["id"]
        v["espa_order"][i] = orderid
        v["espa_archive"][i] = res["archive"]
        v["time"][i] = scene_seconds(r)
        self.n, self.dirty, self.pending = i + 1, True, self.pending + 1
        return i

    def commit(self, status=None):
        ds = self.ds
        t = np.asarray(ds.variables["time"][:], dtype="f8")
        ds.committed_records = np.int32(self.n)
        ds.time_sorted = np.int8(1 if self.n < 2 or bool(np.all(np.diff(t) > 0)) else 0)
        if status:
            ds.processing_status = status
        if self.n:
            ds.time_coverage_start = iso(EPOCH + timedelta(seconds=float(t.min())))
            ds.time_coverage_end = iso(EPOCH + timedelta(seconds=float(t.max())))
        ds.sync()
        self.pending = 0

    def close(self):
        if self.ds is not None:
            self.ds.close()
            self.ds = None


def sort_by_time(path):
    """Rewrite a file with its records in time order (a CF time coordinate must increase)."""
    with nc4.Dataset(path, "r") as src:
        src.set_auto_maskandscale(False)
        t = np.asarray(src.variables["time"][:], dtype="f8")
        if t.size < 2 or np.all(np.diff(t) > 0):
            return False
        order = np.argsort(t, kind="stable")
        tmp = path + ".sorted"
        with nc4.Dataset(tmp, "w", format="NETCDF4") as dst:
            dst.set_auto_maskandscale(False)
            for name, dim in src.dimensions.items():
                dst.createDimension(name, None if dim.isunlimited() else len(dim))
            for name, v in src.variables.items():
                kw = {}
                if v.dtype != str and v.ndim:
                    f = v.filters() or {}
                    ch = v.chunking()
                    kw = dict(zlib=bool(f.get("zlib")), complevel=int(f.get("complevel") or 1), shuffle=bool(f.get("shuffle")),
                              chunksizes=None if ch in (None, "contiguous") else tuple(ch))
                    if "_FillValue" in v.ncattrs():
                        kw["fill_value"] = v.getncattr("_FillValue")
                out = dst.createVariable(name, v.dtype, v.dimensions, **kw)
                out.set_auto_maskandscale(False)      # copy the stored integers; never apply scale_factor on the way
                out.setncatts({k: v.getncattr(k) for k in v.ncattrs() if k != "_FillValue"})
                if "time" not in v.dimensions:
                    if v.ndim == 0:
                        out.assignValue(v.getValue())
                    else:
                        out[:] = v[:]
                else:
                    for i, j in enumerate(order):
                        out[i] = v[int(j)]
            dst.setncatts({k: src.getncattr(k) for k in src.ncattrs()})
            dst.time_sorted = np.int8(1)
    os.replace(tmp, path)
    return True


def readback_check(writer, index, res):
    """Re-read a record from the .nc and compare it bit for bit with what ESPA delivered."""
    for layer in LAYERS:
        got, want = np.asarray(writer.ds.variables[layer][index, :, :]), res["arrays"][layer]
        if got.dtype != want.dtype or not np.array_equal(got, want):
            raise DataIntegrityError(f"{os.path.basename(writer.path)} record {index} {layer}: the file differs "
                                     "from the delivery")


def audit_nc(path):
    """Read-only structural audit of a path/row file. Returns (ok, problems, info)."""
    problems, info = [], {}
    try:
        with nc4.Dataset(path, "r") as ds:
            ds.set_auto_maskandscale(False)
            layers = str(getattr(ds, "layers", "")).split()
            need = ["time", "lat", "lon", "crs", "roi_mask", "roi_fraction", "source_scene"] + layers
            missing = [v for v in need if v not in ds.variables]
            if missing or not layers:
                return False, [f"missing variables {missing}"], info
            n = len(ds.variables["time"])
            t = np.asarray(ds.variables["time"][:], dtype="f8")
            status = str(getattr(ds, "processing_status", "?"))
            if int(getattr(ds, "committed_records", -1)) != n:
                problems.append("record count differs from committed_records")
            if status == "complete" and n > 1 and not np.all(np.diff(t) > 0):
                problems.append("time axis not strictly increasing")
            ids = [str(s) for s in ds.variables["source_scene"][:]] if n else []
            if len(set(ids)) != len(ids):
                problems.append("a scene is stored twice")
            for layer in layers:
                v = ds.variables[layer]
                if v.dimensions != ("time", "lat", "lon") or getattr(v, "grid_mapping", None) != "crs":
                    problems.append(f"{layer}: unexpected dimensions or no grid_mapping")
            if n:
                first = ds.variables[layers[0]]
                empty = sum(1 for i in sorted({0, n // 2, n - 1})
                            if not np.any(np.asarray(first[i, :, :]) != first.getncattr("_FillValue")))
                if empty:
                    problems.append(f"{empty} sampled record(s) hold no value at all")
                info.update(first=iso(EPOCH + timedelta(seconds=float(t.min())))[:10],
                            last=iso(EPOCH + timedelta(seconds=float(t.max())))[:10])
            info.update(records=n, status=status, size=os.path.getsize(path))
    except Exception as e:
        problems.append(f"cannot be opened/read: {type(e).__name__}: {e}")
    return (not problems), problems, info

In [ ]:
# ═══════════════ CELL 9: order -> wait -> download -> write loop, Drive backup, run ═══════════════
class Files:
    """The path/row files of this run on scratch. Drive holds the durable copies."""

    def __init__(self, pathrows):
        self.P = {P["key"]: P for P in pathrows}
        self.w, self.changed, self.next_backup = {}, set(), time.time() + BACKUP_INTERVAL_S

    def paths(self, key):
        return os.path.join(NC_DIR, pr_filename(key)), os.path.join(DRIVE_OUT, pr_filename(key))

    def writer(self, key, meta):
        if key not in self.w:
            local, drv = self.paths(key)
            if not os.path.exists(local):
                if os.path.exists(drv):
                    restore_from_drive(drv, local)
                else:
                    create_pr_nc(local, self.P[key], meta)
            self.w[key] = PRFile(local)
            if str(self.w[key].ds.processing_status) != "in_progress":
                self.w[key].commit(status="in_progress")          # a finished file that receives new scenes
        self.changed.add(key)
        return self.w[key]

    def commit_all(self):
        for w in self.w.values():
            if w.pending:
                w.commit()

    def backup(self, force=False):
        """Verified copy of every changed file to Drive (hourly, and whenever a run stops)."""
        if not self.changed or (not force and time.time() < self.next_backup):
            return
        for key in sorted(self.changed):
            w = self.w[key]
            w.commit()
            w.close()
            try:
                robust_drive_copy(w.path, self.paths(key)[1])
            finally:
                w.open()
        self.changed.clear()
        self.next_backup = time.time() + BACKUP_INTERVAL_S

    def finalize(self, key):
        """All scenes of a path/row are handled: put the records in time order, audit, mark complete."""
        local, drv = self.paths(key)
        w = self.w.pop(key, None)
        if w is None:
            if not os.path.exists(drv):
                return None
            restore_from_drive(drv, local)
            w = PRFile(local)
        w.commit()
        w.close()
        sort_by_time(local)
        good, problems, info = audit_nc(local)
        w = PRFile(local)
        w.commit(status="complete" if good else "in_progress")
        w.close()
        if not good:
            log.error("[%s] AUDIT FAILED - left as in_progress: %s", key, problems)
        robust_drive_copy(local, drv)
        os.remove(local)
        self.changed.discard(key)
        summary = dict(key=key, status="complete" if good else "audit failed", records=info.get("records", 0),
                       size=os.path.getsize(drv), problems=problems)
        log_events(LOG_FILES, [summary])
        log.info("[%s] DONE: %d scenes, %s on Drive", key, summary["records"], human(summary["size"]))
        return summary


def ingest(led, ready, by_scene, files, checks):
    """Download the finished scenes (several at a time), read each onto the grid, append, re-read."""
    written = 0
    with timed("download + read + write"), ThreadPoolExecutor(N_DOWNLOAD_THREADS) as ex:
        futs = {ex.submit(download_item, scene, item): (scene, item) for scene, item in ready}
        for fut in as_completed(futs):
            scene, item = futs[fut]
            try:
                d = fut.result()
            except NotThere as e:
                led["scenes"][scene] = dict(state="reorder", note=str(e), order=item["order"])
                continue
            except Exception as e:
                log.warning("%s: %s - tried again on the next round", scene, str(e)[:200])
                continue
            P, r = by_scene[scene]
            res = read_delivery(d["path"], P)                # a grid or content problem stops the run on purpose
            w = files.writer(P["key"], res["meta"])
            i = w.append(r, res, item["order"])
            if checks["left"] > 0:
                readback_check(w, i, res)
                checks["left"] -= 1
            if w.pending >= 100:
                w.commit()
            if res["stats"]["outside_window"]:
                log.warning("%s: %d valid cells lie outside this path/row's window and were not stored", scene,
                            res["stats"]["outside_window"])
            led["scenes"][scene] = dict(state="written", order=item["order"])
            TOTALS["bytes"] = TOTALS.get("bytes", 0) + d["bytes"]
            TOTALS["md5_checked"] = TOTALS.get("md5_checked", 0) + int(d["md5_checked"])
            os.remove(d["path"])
            written += 1
    files.commit_all()
    save_ledger(led)
    return written


def run_pipeline():
    t0 = time.time()
    TIMES.clear()
    TOTALS.clear()
    log.info("=" * 78)
    log.info("  Landsat indices from USGS ESPA at 4 km | %s | resampling %s | TEST_MODE=%s", " ".join(LAYERS),
             RESAMPLING, TEST_MODE)
    log.info("  output: %s", DRIVE_OUT)
    log.info("=" * 78)
    files, stopped, summaries, led, counts = None, None, [], None, {}
    try:
        espa = Espa(ESPA_API, ers_credentials())
        espa.user()                                              # fails here, clearly, if the login is refused
        log.info("ESPA login OK")
        ensure_disk_free("start")
        shutil.rmtree(NC_DIR, ignore_errors=True)                # scratch copies are never trusted across runs
        os.makedirs(NC_DIR, exist_ok=True)
        pathrows = build_pathrows(get_manifest())
        if ONLY_PATHROWS:
            pathrows = [P for P in pathrows if P["key"] in set(ONLY_PATHROWS)]
        if TEST_MODE and pathrows:
            pathrows = pathrows[:2]
            per = max(1, TEST_SCENES // len(pathrows))
            for P in pathrows:
                pick = sorted({int(round(v)) for v in np.linspace(0, len(P["scenes"]) - 1, min(per, len(P["scenes"])))})
                P["scenes"] = [P["scenes"][i] for i in pick]
        if not pathrows:
            raise PipelineStop("No path/row to process - check SATELLITES, the dates and ONLY_PATHROWS.")
        by_scene = {r["id"]: (P, r) for P in pathrows for r in P["scenes"]}
        order_of = list(by_scene)                                # path/row by path/row, each in time order
        log.info("%d path/rows, %d scenes", len(pathrows), len(by_scene))

        # ---------- what is already safe: the scenes inside the files on Drive ----------
        led = load_ledger()
        on_drive, complete_on_drive = set(), set()
        files = Files(pathrows)
        for P in pathrows:
            drv = files.paths(P["key"])[1]
            if os.path.exists(drv):
                st = file_state(drv)
                if st is None:
                    aside = f"{drv}.unreadable_{utcnow():%Y%m%dT%H%M%S}"
                    os.replace(drv, aside)
                    log.error("[%s] the Drive copy could not be trusted - moved aside as %s", P["key"], os.path.basename(aside))
                else:
                    on_drive |= st["scenes"]
                    if st["status"] == "complete":
                        complete_on_drive.add(P["key"])
        for s in on_drive & set(by_scene):
            led["scenes"][s] = dict(led["scenes"].get(s, {}), state="written")
        for s, rec in led["scenes"].items():                     # written according to the ledger, but not on Drive
            if rec["state"] == "written" and s not in on_drive and s in by_scene:
                rec["state"] = "ordered" if rec.get("order") in led["orders"] else "reorder"
                if rec["state"] == "ordered":
                    led["orders"][rec["order"]]["closed"] = False
        save_ledger(led)

        checks = dict(left=len(by_scene) if TEST_MODE else READBACK_CHECKS * len(pathrows))
        finished = set()
        deadline = t0 + MAX_RUN_HOURS * 3600
        while True:
            state = {s: led["scenes"].get(s, {}).get("state") for s in order_of}
            # ---------- 1. order what is not ordered yet, within ESPA's limit of open scenes ----------
            todo = [s for s in order_of if state[s] in (None, "reorder")]
            n_open = sum(1 for v in led["scenes"].values() if v["state"] == "ordered")
            while todo and n_open + min(len(todo), SCENES_PER_ORDER) <= MAX_OPEN_SCENES:
                batch, todo = todo[:SCENES_PER_ORDER], todo[SCENES_PER_ORDER:]
                note = f"{PRODUCT_TAG} {ROI_LABEL} {batch[0][:4]} {len(batch)} scenes"
                if submit_order(espa, led, batch, note):
                    n_open += sum(1 for s in batch if led["scenes"][s]["state"] == "ordered")
            # ---------- 2. ask ESPA how the open orders are doing ----------
            ready, counts = poll_orders(espa, led)
            # ---------- 3. download and write what is finished ----------
            n_new = 0
            if ready:
                ensure_disk_free("download")
                n_new = ingest(led, ready, by_scene, files, checks)
                files.backup()
            # ---------- 4. close the path/rows whose scenes are all handled ----------
            for P in pathrows:
                if P["key"] not in finished and all(led["scenes"].get(r["id"], {}).get("state") in ("written", "unavailable")
                                                    for r in P["scenes"]):
                    finished.add(P["key"])
                    if P["key"] in complete_on_drive and P["key"] not in files.w:
                        continue                                 # finished in an earlier run and untouched since
                    s = files.finalize(P["key"])
                    if s:
                        summaries.append(s)
            # ---------- 5. report, then stop or wait ----------
            tally = {}
            for s in order_of:
                k = led["scenes"].get(s, {}).get("state") or "not ordered yet"
                tally[k] = tally.get(k, 0) + 1
            log.info("scenes: %s | at ESPA: %s | %s downloaded", tally, counts or "-", human(TOTALS.get("bytes", 0)))
            if len(finished) == len(pathrows):
                break
            if time.time() > deadline:
                raise PipelineStop(f"MAX_RUN_HOURS reached with {tally.get('ordered', 0)} scenes still at ESPA and "
                                   f"{tally.get('not ordered yet', 0) + tally.get('reorder', 0)} not ordered yet. The "
                                   "orders keep running on USGS's side - run all again later to collect them.")
            if not n_new:
                time.sleep(POLL_MINUTES * 60)
    except PipelineStop as e:
        stopped = e
    finally:
        if files is not None:
            try:
                files.backup(force=True)                         # Drive ends up with everything written so far
                for w in files.w.values():
                    w.close()
            except Exception as e:
                log.error("Final backup failed: %s", e)
        if led is not None:
            try:
                save_ledger(led)
            except Exception as e:
                log.error("Ledger could not be saved: %s", e)

    log.info("=" * 78)
    if stopped is not None:
        print("\n" + "=" * 78 + f"\n  STOPPED (nothing is lost, re-run to resume)\n  {stopped}\n" + "=" * 78 + "\n")
    n_written = sum(1 for v in (led or {}).get("scenes", {}).values() if v["state"] == "written")
    log.info("  %d scenes written in total, %d path/row files completed in this run, %s downloaded, %.1f min",
             n_written, len(summaries), human(TOTALS.get("bytes", 0)), (time.time() - t0) / 60)
    lost = [(s, v.get("note", "")) for s, v in (led or {}).get("scenes", {}).items() if v["state"] == "unavailable"]
    if lost:
        log.info("  %d scene(s) ESPA could not deliver (see %s); first: %s - %s", len(lost), LOG_FAILED, lost[0][0], lost[0][1][:120])
    if TIMES:
        log.info("  time by stage: %s", ", ".join(f"{k} {v:.0f} s" for k, v in sorted(TIMES.items(), key=lambda kv: -kv[1])))
    if TEST_MODE and n_written and TOTALS.get("bytes"):
        per = TOTALS["bytes"] / max(1, n_written)
        log.info("  PROJECTION (rough): about %s per scene to download, so roughly %s for 65,000 scenes; MD5 checked "
                 "on %d of %d archives. ESPA took %.0f min for this order; the whole record is about %d orders of %d.",
                 human(per), human(per * 65000), TOTALS.get("md5_checked", 0), n_written, (time.time() - t0) / 60,
                 math.ceil(65000 / SCENES_PER_ORDER), SCENES_PER_ORDER)
    log.info("=" * 78)
    return summaries


RESULTS = run_pipeline()

In [ ]:
# ═══════════════ CELL 10: audit every path/row file on Drive ═══════════════
def audit_all():
    paths = sorted(glob.glob(os.path.join(DRIVE_OUT, f"{PRODUCT_TAG}_p*_{ROI_LABEL}.nc")))
    print(f"{len(paths)} path/row files in {DRIVE_OUT}\n")
    print(f"{'file':50s} {'status':12s} {'scenes':>7s} {'size':>10s}  first .. last scene")
    total, bad = 0, 0
    for p in paths:
        ok, problems, info = audit_nc(p)
        total += info.get("size", 0)
        print(f"{os.path.basename(p):50s} {info.get('status', '?'):12s} {info.get('records', 0):7d} "
              f"{human(info.get('size', 0)):>10s}  {info.get('first', '-')} .. {info.get('last', '-')}")
        if not ok:
            bad += 1
            for msg in problems:
                print(f"      PROBLEM: {msg}")
    print(f"\nTotal {human(total)} | {bad} file(s) with problems")
    for name in (LOG_FAILED, LOG_SKIPPED):
        path = os.path.join(DRIVE_OUT, name)
        n = sum(1 for _ in open(path)) if os.path.exists(path) else 0
        print(f"{name}: {n} entries")


audit_all()

# Reading a file (xarray applies the scale factor and the fill value by itself):
#   import xarray as xr
#   ds = xr.open_dataset("/content/drive/MyDrive/MSUGWB/LANDSAT_ESPA_indices_4km/<file>.nc")
#   clear = (ds.qa_pixel.astype("uint16") & 0b111110) == 0     # the sampled pixel is not cloud, shadow or snow
#   ndvi = ds.ndvi.where(clear & (ds.ndvi <= 1))               # also drops the saturation code (2.0)
#
# Optional independent CF check of one finished file:
# !pip -q install compliance-checker
# !compliance-checker --test=cf:1.9 "/content/drive/MyDrive/MSUGWB/LANDSAT_ESPA_indices_4km/<file>.nc"